## GraphDB QA with ontology-guided SPARQL (Gemma 3 12B)

Simple pipeline:

1. User asks a natural-language question.
2. `gpt-oss:120b` generates SPARQL using a compact summary from `ontology.ttl`.
3. We execute the SPARQL query against GraphDB.
4. The same `gpt-oss:120b` answers from GraphDB results only.

In [1]:
# Install once if needed
!pip install -q requests rdflib openai

In [9]:
from pathlib import Path
import json
import os
import re
import requests
from rdflib import Graph, RDF, RDFS, OWL, URIRef
from openai import OpenAI

project_root = Path('.').resolve()
ontology_path = project_root / 'ontology.ttl'

# ---- GraphDB config ----
GRAPHDB_BASE = 'http://localhost:7200'
REPOSITORY_ID = 'Master_Thesis'
SPARQL_ENDPOINT = f"{GRAPHDB_BASE}/repositories/{REPOSITORY_ID}"

# ---- LLM (Open WebUI, OpenAI-compatible) ----
LLM_BASE_URL = os.getenv('LLM_BASE_URL', 'https://web.ollama-gpt-oss.ai.wu.ac.at/api')
LLM_API_KEY = os.getenv('LLM_API_KEY', 'sk-9b41b856cc0b403b8a3c10618f1c2996')
LLM_MODEL = 'gpt-oss:120b'

llm_client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    timeout=120.0,
    max_retries=4,
)

print('Project root:', project_root)
print('ontology.ttl exists:', ontology_path.exists())
print('GraphDB endpoint:', SPARQL_ENDPOINT)
print('LLM base URL:', LLM_BASE_URL)
print('LLM model:', LLM_MODEL)

Project root: C:\Users\dyury\Desktop\Master Thesis
ontology.ttl exists: True
GraphDB endpoint: http://localhost:7200/repositories/Master_Thesis
LLM base URL: https://web.ollama-gpt-oss.ai.wu.ac.at/api
LLM model: gpt-oss:120b


In [3]:
# Local merged KG (import this file into GraphDB for instance queries)
kg_ttl_path = project_root / 'fcdf_kg.ttl'
print('Local KG file (import into GraphDB):', kg_ttl_path)
print('Exists:', kg_ttl_path.exists())
if kg_ttl_path.exists():
    mb = kg_ttl_path.stat().st_size / (1024 * 1024)
    print(f'Size (MB): {mb:.2f}')
print('Note: cells below query GraphDB only, not this path.')

Local KG file (import into GraphDB): C:\Users\dyury\Desktop\Master Thesis\fcdf_kg.ttl
Exists: True
Size (MB): 111.73
Note: cells below query GraphDB only, not this path.


In [ ]:
_CORE_NS = 'https://w3id.org/football-cdf/core#'

#  collapses long absolute URIs into the prefixed forms the LLM is expected to write in SPARQL
def _short(uri: str) -> str:
    if uri.startswith(_CORE_NS):
        return 'core:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/2001/XMLSchema#'):
        return 'xsd:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/2000/01/rdf-schema#'):
        return 'rdfs:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/1999/02/22-rdf-syntax-ns#'):
        return 'rdf:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/2002/07/owl#'):
        return 'owl:' + uri.split('#', 1)[1]
    return f'<{uri}>'

# Return short names for an rdfs:domain/range value, expanding owl:unionOf lists.
def _expand_domain_or_range(g: Graph, node) -> list[str]:
    out: list[str] = []
    if isinstance(node, URIRef):
        out.append(_short(str(node)))
        return out
    for u in g.objects(node, OWL.unionOf):
        cur = u
        while cur is not None and cur != RDF.nil:
            for first in g.objects(cur, RDF.first):
                if isinstance(first, URIRef):
                    out.append(_short(str(first)))
            rests = list(g.objects(cur, RDF.rest))
            cur = rests[0] if rests else None
    return out

# Loads ontology.ttl into an in-memory rdflib graph.
# Build a richer schema hint for SPARQL generation.
# Includes classes plus object/datatype properties with their declared
# rdfs:domain, rdfs:range, and short rdfs:comment when available.
# max_items=0 means no truncation; positive values cap each section.
def load_ontology_schema_for_prompt(path: Path, max_items: int = 0) -> str:
    g = Graph()
    g.parse(path, format='turtle')

    classes: list[str] = []
    for s in sorted(set(g.subjects(RDF.type, OWL.Class)), key=str):
        if isinstance(s, URIRef) and str(s).startswith(_CORE_NS):
            classes.append(_short(str(s)))

# this allows model to see exactly which subject a property is allowed on, which prevents misuse -> core:<name> | domain: core:<X> | range: <Y> -- <short comment>
    def gather_props(p_type) -> list[str]:
        rows: list[str] = []
        for p in sorted(set(g.subjects(RDF.type, p_type)), key=str):
            if not isinstance(p, URIRef):
                continue
            p_uri = str(p)
            if not p_uri.startswith(_CORE_NS):
                continue
            doms: list[str] = []
            for d in g.objects(p, RDFS.domain):
                doms.extend(_expand_domain_or_range(g, d))
            rngs: list[str] = []
            for r in g.objects(p, RDFS.range):
                rngs.extend(_expand_domain_or_range(g, r))
            doms = sorted(set(doms))
            rngs = sorted(set(rngs))
            comment = next((str(c) for c in g.objects(p, RDFS.comment)), '').strip().replace('\n', ' ')
            if len(comment) > 90:
                comment = comment[:87] + '...'
            parts = [_short(p_uri)]
            if doms:
                parts.append(f"domain: {', '.join(doms)}")
            if rngs:
                parts.append(f"range: {', '.join(rngs)}")
            line = ' | '.join(parts)
            if comment:
                line += f' -- {comment}'
            rows.append(line)
        return rows

    obj_props = gather_props(OWL.ObjectProperty)
    data_props = gather_props(OWL.DatatypeProperty)

    if max_items and max_items > 0:
        classes = classes[:max_items]
        obj_props = obj_props[:max_items]
        data_props = data_props[:max_items]

    sections = []
    if classes:
        sections.append('Classes:\n' + '\n'.join('- ' + c for c in classes))
    if obj_props:
        sections.append('Object properties:\n' + '\n'.join('- ' + p for p in obj_props))
    if data_props:
        sections.append('Datatype properties:\n' + '\n'.join('- ' + p for p in data_props))
    return '\n\n'.join(sections)


DATA_GRAPH_HINT = """
Instance data (fcdf_kg.ttl) often uses:
- core:events_goals, core:events_cards, core:events_subtitutions (subproperties of core:events from Match to Goal / Card / Subtitution)
- core:related_event_ids (links between Event resources)
- core:pass_outcome_type, core:pass_type (Passes); core:shot_outcome_type, core:shot_type (Shots / Goals)
""".strip()

schema_hint = load_ontology_schema_for_prompt(ontology_path) + "\n\n" + DATA_GRAPH_HINT
print('\n'.join(schema_hint.splitlines()[:20]))
if len(schema_hint.splitlines()) > 20:
    print('...')

Classes:
- core:Card
- core:Competition
- core:Event
- core:Goal
- core:Match
- core:Match_Result
- core:Match_Status
- core:Meta
- core:Misc
- core:Pass
- core:Player
- core:Referee
- core:Season
- core:Shot
- core:Subtitution
- core:Team
- core:Vendor
- core:Whistle

...


In [ ]:
def call_llm(prompt: str, model: str = LLM_MODEL) -> str:
    resp = llm_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
    )
    return (resp.choices[0].message.content or '').strip()


def extract_sparql(text: str) -> str:
    # Prefer fenced ```sparql blocks if present.
    match = re.search(r"```(?:sparql)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text.strip()


try:
    from rdflib.plugins.sparql.parser import parseQuery
except ImportError:
    parseQuery = None  # type: ignore[misc, assignment]


# Rewrite invalid 4-token triples ?S core:P rdfs:label|comment ?O into one property path.
def fix_invalid_core_then_rdfs_label(q: str) -> str:
    for rdfp in ("rdfs:label", "rdfs:comment"):
        q = re.sub(
            rf"(?P<subj>\?\w+)\s+(?P<core>core:[A-Za-z_][A-Za-z0-9_]*)\s+{rdfp}\s+(?P<obj>\?\w+)\s*\.",
            rf"\g<subj> \g<core>/{rdfp} \g<obj> .",
            q,
        )
        q = re.sub(
            rf"(?P<subj>\?\w+)\s+(?P<core>core:[A-Za-z_][A-Za-z0-9_]*)\s+{rdfp}\s+(?P<lit>\"(?:[^\"\\\\]|\\\\.)*\")\s*\.",
            rf"\g<subj> \g<core>/{rdfp} \g<lit> .",
            q,
        )
    return q


def normalize_sparql(query: str) -> str:
    q = query.strip()

    # Fix malformed 'PREFIX:' pattern into a real prefix declaration.
    q = re.sub(
        r"(?im)^\s*PREFIX\s*:\s*<https?://w3id\.org/football-cdf/core#>\s*$",
        "PREFIX core: <https://w3id.org/football-cdf/core#>",
        q,
    )

    # Ensure prefix declarations exist when prefixed names are used.
    if re.search(r"\bcore:[A-Za-z_]", q) and 'PREFIX core:' not in q:
        q = "PREFIX core: <https://w3id.org/football-cdf/core#>\n" + q

    if re.search(r"\bxsd:[A-Za-z_]", q) and 'PREFIX xsd:' not in q:
        q = "PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\n" + q

    if re.search(r"\brdfs:[A-Za-z_]", q) and 'PREFIX rdfs:' not in q:
        q = "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n" + q

    # Align with existing data where ids are typically string literals.
    q = re.sub(r'"(\d+)"\^\^xsd:integer', r'"\1"', q)

    # Fix common bad UNION layout: WHERE { block1 } UNION { block2 }
    union_outside_where = re.search(
        r"(?is)(.*?SELECT\s+.*?WHERE\s*)\{\s*(.*?)\s*\}\s*UNION\s*\{\s*(.*?)\s*\}\s*$",
        q,
    )
    if union_outside_where:
        prefix_select = union_outside_where.group(1).strip()
        block_1 = union_outside_where.group(2).strip()
        block_2 = union_outside_where.group(3).strip()
        q = (
            f"{prefix_select}{{\n"
            f"  {{\n{block_1}\n  }}\n"
            f"  UNION\n"
            f"  {{\n{block_2}\n  }}\n"
            f"}}"
        )

    q = fix_invalid_core_then_rdfs_label(q)
    return q


_ALLOWED_CORE_TERMS_CACHE = None


# Build an allowlist of football-cdf core terms from ontology.ttl.
# Cached so we do not re-parse ontology on every question.
def get_allowed_core_terms() -> set[str]:
    global _ALLOWED_CORE_TERMS_CACHE
    if _ALLOWED_CORE_TERMS_CACHE is not None:
        return _ALLOWED_CORE_TERMS_CACHE

    g = Graph()
    g.parse(ontology_path, format='turtle')

    q = """
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>

    SELECT ?term WHERE {
      {
        ?term rdf:type owl:Class .
      }
      UNION
      {
        ?term rdf:type owl:ObjectProperty .
      }
      UNION
      {
        ?term rdf:type owl:DatatypeProperty .
      }
    }
    """

    allowed = set()
    for row in g.query(q):
        term = str(row.term)
        if term.startswith('https://w3id.org/football-cdf/core#'):
            allowed.add(term.split('#', 1)[1])

    _ALLOWED_CORE_TERMS_CACHE = allowed
    return allowed


def validate_core_terms(query: str) -> tuple[bool, str]:
    allowed = get_allowed_core_terms()
    used = set(re.findall(r'\bcore:([A-Za-z_][A-Za-z0-9_]*)\b', query))
    unknown = sorted(t for t in used if t not in allowed)
    if unknown:
        preview = ', '.join(unknown[:8])
        if len(unknown) > 8:
            preview += ', ...'
        return False, f'Unknown core: terms not present in ontology: {preview}'
    return True, 'ok'


_PROP_DOMAIN_CACHE: dict[str, set[str]] | None = None


# Map core property local-name -> set of declared domain class local-names.
# Properties without rdfs:domain are absent from the map (treated as universal).
# Handles owl:unionOf domain declarations.
def _load_property_domains() -> dict[str, set[str]]:
    global _PROP_DOMAIN_CACHE
    if _PROP_DOMAIN_CACHE is not None:
        return _PROP_DOMAIN_CACHE
    g = Graph()
    g.parse(ontology_path, format='turtle')
    domains: dict[str, set[str]] = {}
    for p, _, d in g.triples((None, RDFS.domain, None)):
        if not isinstance(p, URIRef) or not str(p).startswith(_CORE_NS):
            continue
        p_local = str(p).split('#', 1)[1]
        classes: list[str] = []
        if isinstance(d, URIRef):
            if str(d).startswith(_CORE_NS):
                classes.append(str(d).split('#', 1)[1])
        else:
            for u in g.objects(d, OWL.unionOf):
                cur = u
                while cur is not None and cur != RDF.nil:
                    for first in g.objects(cur, RDF.first):
                        if isinstance(first, URIRef) and str(first).startswith(_CORE_NS):
                            classes.append(str(first).split('#', 1)[1])
                    rests = list(g.objects(cur, RDF.rest))
                    cur = rests[0] if rests else None
        if classes:
            domains.setdefault(p_local, set()).update(classes)
    _PROP_DOMAIN_CACHE = domains
    return _PROP_DOMAIN_CACHE


def validate_property_domains(query: str) -> tuple[bool, str]:
    domains = _load_property_domains()
    var_class: dict[str, set[str]] = {}
    for m in re.finditer(
        r'\?([A-Za-z_][A-Za-z0-9_]*)\s+(?:a|rdf:type)\s+core:([A-Za-z_][A-Za-z0-9_]*)',
        query,
    ):
        var_class.setdefault(m.group(1), set()).add(m.group(2))

    violations: list[str] = []
    for m in re.finditer(
        r'\?([A-Za-z_][A-Za-z0-9_]*)\s+core:([A-Za-z_][A-Za-z0-9_]*)\b',
        query,
    ):
        var, prop = m.group(1), m.group(2)
        if var not in var_class:
            continue
        prop_domains = domains.get(prop)
        if not prop_domains:
            continue
        if not (var_class[var] & prop_domains):
            violations.append(
                f"core:{prop} (domain: {', '.join(sorted(prop_domains))}) used on "
                f"?{var} (typed as core:{', core:'.join(sorted(var_class[var]))})"
            )

    if violations:
        violations = sorted(set(violations))
        return False, 'Property used on wrong subject class: ' + '; '.join(violations[:4])
    return True, 'ok'


def is_likely_valid_sparql(query: str) -> tuple[bool, str]:
    q = query.strip()
    q_upper = q.upper()

    if not q:
        return False, 'Empty query.'
    if 'SELECT' not in q_upper:
        return False, 'Only SELECT queries are supported in this notebook.'
    if q.count('{') != q.count('}'):
        return False, 'Unbalanced braces in query.'
    if re.search(r'(?im)^\s*PREFIX\s*:', q):
        return False, 'Malformed PREFIX declaration. Use e.g. PREFIX core: <...>.'

    # Heuristic: UNION must be followed by a grouped block (after optional whitespace).
    if re.search(r'\bUNION\b(?!\s*\{)', q_upper):
        return False, 'UNION must be followed by a grouped block: UNION { ... }'

    ok_terms, reason_terms = validate_core_terms(q)
    if not ok_terms:
        return False, reason_terms

    ok_dom, reason_dom = validate_property_domains(q)
    if not ok_dom:
        return False, reason_dom

    if parseQuery is not None:
        try:
            parseQuery(q)
        except Exception as e:
            msg = str(e).replace("\n", " ")
            if len(msg) > 220:
                msg = msg[:220] + "..."
            return False, f"SPARQL parse error: {msg}"

    return True, 'ok'


def generate_sparql(question: str, schema: str) -> str:
    prompt = f"""
You create SPARQL SELECT queries for GraphDB.

Use only terms from this ontology summary:
{schema}

Rules:
- Return exactly ONE executable SPARQL SELECT query.
- Use explicit prefixes when needed, for example:
  PREFIX core: <https://w3id.org/football-cdf/core#>
  PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
- Never output malformed PREFIX syntax like 'PREFIX: <...>'.
- Keep query syntax strict and valid for GraphDB.
- Respect each property's declared `domain:` from the schema above. Never use
  a property on a subject of an unrelated class (e.g. core:has_played has
  domain core:Player, so it must NOT appear on a ?match typed as core:Match).
- Each basic triple pattern is exactly THREE terms then a dot: subject predicate object .
  NEVER write four terms on one line (e.g. ?m core:teams_home rdfs:label ?n is INVALID).
- To read a label after an object property (team, competition, season, referee), use either TWO triples:
  ?m core:teams_home ?t . ?t rdfs:label ?name .
  OR one property path: ?m core:teams_home/rdfs:label ?name .
  Same idea for core:teams_away, core:competition, core:season, core:referee.
- If you use UNION, each branch MUST be wrapped in braces INSIDE one outer WHERE block:
  WHERE {{ {{ ... }} UNION {{ ... }} }}
- For team names, prefer rdfs:label on the Team resource (not core:name on the match).
- Do not output any explanation or markdown, only the query.
- If unsure, still produce a best-effort valid SELECT query.

User question:
{question}
    """.strip()

    raw = call_llm(prompt)
    return normalize_sparql(extract_sparql(raw))


def repair_sparql(question: str, bad_query: str, error_hint: str, schema: str) -> str:
    prompt = f"""
Fix this SPARQL query so it is valid and executable in GraphDB.

Question:
{question}

Ontology summary:
{schema}

Validation error:
{error_hint}

Bad query:
{bad_query}

Rules:
- Return exactly one corrected SPARQL SELECT query.
- Keep the intent of the original question.
- Each triple pattern must be subject predicate object only (three terms before the dot).
  Never chain rdfs:label as a second predicate on the same line as core:teams_home (use ?m core:teams_home/rdfs:label ?n or two triples).
- If UNION is used, format as {{ ... }} UNION {{ ... }}.
- Use proper prefix declarations (e.g., PREFIX core: <...>).
- Output query only, no explanation.
    """.strip()

    return normalize_sparql(extract_sparql(call_llm(prompt)))


def run_sparql(query: str, endpoint: str = SPARQL_ENDPOINT) -> dict:
    headers = {'Accept': 'application/sparql-results+json'}
    resp = requests.get(endpoint, params={'query': query}, headers=headers, timeout=720)
    resp.raise_for_status()
    return resp.json()


def run_sparql_safe(
    question: str,
    query: str,
    schema: str,
    endpoint: str = SPARQL_ENDPOINT,
    max_repair_attempts: int = 2,
) -> tuple[str, dict]:
    """
    Robust generic loop:
    - normalize
    - validate syntax + ontology terms
    - attempt execution
    - if failing, repair with explicit error feedback and retry
    """
    current_query = normalize_sparql(query)
    last_error = None

    for _ in range(max_repair_attempts + 1):
        ok, reason = is_likely_valid_sparql(current_query)
        if not ok:
            last_error = f'Validation failed: {reason}'
            current_query = normalize_sparql(repair_sparql(question, current_query, last_error, schema))
            continue

        try:
            result_json = run_sparql(current_query, endpoint=endpoint)
            bindings = result_json.get('results', {}).get('bindings', [])
            if bindings:
                return current_query, result_json

            # Query was valid but likely too restrictive/wrong relation: repair once using result feedback.
            last_error = 'Query executed but returned no rows. Try a semantically close alternative relation from ontology.'
            current_query = normalize_sparql(repair_sparql(question, current_query, last_error, schema))
        except requests.HTTPError as e:
            last_error = f'GraphDB HTTP error: {e}'
            current_query = normalize_sparql(repair_sparql(question, current_query, last_error, schema))

    raise RuntimeError(f'Unable to produce executable SPARQL after retries. Last error: {last_error}')


def format_bindings(result_json: dict, max_rows: int = 30) -> str:
    head_vars = result_json.get('head', {}).get('vars', [])
    bindings = result_json.get('results', {}).get('bindings', [])
    if not bindings:
        return 'No rows returned.'

    lines = []
    for i, row in enumerate(bindings[:max_rows], start=1):
        vals = []
        for v in head_vars:
            vals.append(f"{v}={row.get(v, {}).get('value', '')}")
        lines.append(f"{i}. " + '; '.join(vals))
    if len(bindings) > max_rows:
        lines.append(f"... ({len(bindings) - max_rows} more rows)")
    return '\n'.join(lines)


def answer_from_results(question: str, query: str, result_json: dict) -> str:
    table_text = format_bindings(result_json)
    prompt = f"""
You are answering a user question using only GraphDB query results.

Question:
{question}

SPARQL query used:
{query}

Query results:
{table_text}

Rules:
- Use only the provided results.
- If results are empty or insufficient, say you do not know.
- Be concise.
    """.strip()

    return call_llm(prompt)


def graphdb_qa(question: str) -> dict:
    initial_sparql = generate_sparql(question, schema_hint)
    final_sparql, result_json = run_sparql_safe(question, initial_sparql, schema_hint)
    answer = answer_from_results(question, final_sparql, result_json)
    return {
        'question': question,
        'initial_sparql': initial_sparql,
        'sparql': final_sparql,
        'result_json': result_json,
        'answer': answer,
    }

In [11]:
# Example question (always print query first)
question = 'Which teams played in match 3895052?'

print('QUESTION:\n', question)

# Step 1: generate query and print it even if later steps fail
initial_sparql = generate_sparql(question, schema_hint)
print('\nINITIAL SPARQL GENERATED:\n', initial_sparql)

# Keep a stable result object for downstream debug cells.
result = {
    'question': question,
    'initial_sparql': initial_sparql,
    'sparql': initial_sparql,
    'result_json': None,
    'answer': None,
    'error': None,
}

try:
    # Step 2: validate/repair and execute against GraphDB
    final_sparql, result_json = run_sparql_safe(question, initial_sparql, schema_hint)
    result['sparql'] = final_sparql
    result['result_json'] = result_json
    print('\nFINAL SPARQL EXECUTED:\n', final_sparql)

    # Step 3: answer from DB results
    answer = answer_from_results(question, final_sparql, result_json)
    result['answer'] = answer
    print('\nANSWER:\n', answer)
except Exception as e:
    result['error'] = str(e)
    print('\nGraphDB execution failed:', e)
    print('\nKeep this query for debugging (generated before failure):\n', initial_sparql)

QUESTION:
 Which teams played in match 3895052?

INITIAL SPARQL GENERATED:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?homeTeam ?awayTeam WHERE {
  ?match a core:Match .
  ?match core:id "3895052" .
  ?match core:teams_home/rdfs:label ?homeTeam .
  ?match core:teams_away/rdfs:label ?awayTeam .
}

FINAL SPARQL EXECUTED:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?homeTeam ?awayTeam WHERE {
  ?match a core:Match .
  ?match core:id "3895052" .
  ?match core:teams_home/rdfs:label ?homeTeam .
  ?match core:teams_away/rdfs:label ?awayTeam .
}

ANSWER:
 The match featured Bayer Leverkusen (home) against RB Leipzig (away).


In [ ]:
# Example question (always print query first)
question = 'how many matches were played in the tournament?'

print('QUESTION:\n', question)

# Step 1: generate query and print it even if later steps fail
initial_sparql = generate_sparql(question, schema_hint)
print('\nINITIAL SPARQL GENERATED:\n', initial_sparql)

# Keep a stable result object for downstream debug cells.
result = {
    'question': question,
    'initial_sparql': initial_sparql,
    'sparql': initial_sparql,
    'result_json': None,
    'answer': None,
    'error': None,
}

try:
    # Step 2: validate/repair and execute against GraphDB
    final_sparql, result_json = run_sparql_safe(question, initial_sparql, schema_hint)
    result['sparql'] = final_sparql
    result['result_json'] = result_json
    print('\nFINAL SPARQL EXECUTED:\n', final_sparql)

    # Step 3: answer from DB results
    answer = answer_from_results(question, final_sparql, result_json)
    result['answer'] = answer
    print('\nANSWER:\n', answer)
except Exception as e:
    result['error'] = str(e)
    print('\nGraphDB execution failed:', e)
    print('\nKeep this query for debugging (generated before failure):\n', initial_sparql)

QUESTION:
 how many matches were played in the tournament?

INITIAL SPARQL GENERATED:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (COUNT(DISTINCT ?m) AS ?matchCount)
WHERE {
  ?m a core:Match .
}

FINAL SPARQL EXECUTED:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (COUNT(DISTINCT ?m) AS ?matchCount)
WHERE {
  ?m a core:Match .
}

ANSWER:
 The tournament included **34 matches**.


In [ ]:
# Example question (always print query first)
question = 'List me all the matches and teams that played in the cronological order. please Include scores'

print('QUESTION:\n', question)

# Step 1: generate query and print it even if later steps fail
initial_sparql = generate_sparql(question, schema_hint)
print('\nINITIAL SPARQL GENERATED:\n', initial_sparql)

# Keep a stable result object for downstream debug cells.
result = {
    'question': question,
    'initial_sparql': initial_sparql,
    'sparql': initial_sparql,
    'result_json': None,
    'answer': None,
    'error': None,
}

try:
    # Step 2: validate/repair and execute against GraphDB
    final_sparql, result_json = run_sparql_safe(question, initial_sparql, schema_hint)
    result['sparql'] = final_sparql
    result['result_json'] = result_json
    print('\nFINAL SPARQL EXECUTED:\n', final_sparql)

    # Step 3: answer from DB results
    answer = answer_from_results(question, final_sparql, result_json)
    result['answer'] = answer
    print('\nANSWER:\n', answer)
except Exception as e:
    result['error'] = str(e)
    print('\nGraphDB execution failed:', e)
    print('\nKeep this query for debugging (generated before failure):\n', initial_sparql)

QUESTION:
 List me all the matches and teams that played in the cronological order. please Include scores

INITIAL SPARQL GENERATED:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?matchId ?kickoff ?homeTeam ?awayTeam ?homeScore ?awayScore
WHERE {
  ?match a core:Match .
  ?match core:id ?matchId .
  ?match core:kickoff_time ?kickoff .
  ?match core:teams_home ?home .
  ?home rdfs:label ?homeTeam .
  ?match core:teams_away ?away .
  ?away rdfs:label ?awayTeam .
  ?match core:match_result ?result .
  ?result core:result_home ?homeScore .
  ?result core:result_away ?awayScore .
}
ORDER BY ?kickoff

FINAL SPARQL EXECUTED:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?matchId ?kickoff ?homeTeam ?awayTeam ?homeScore ?awayScore
WHERE {
  ?match a core:Match .
  ?mat